# Fase 2 - Parte 1: Extracao de Sintomas e Diagnostico Assistido

Este notebook implementa a primeira parte da atividade principal da Fase 2 do CardioIA.

Objetivos desta etapa:
- ler frases simuladas de pacientes;
- consultar um mapa de conhecimento em CSV;
- identificar sintomas por correspondencia textual;
- sugerir um diagnostico assistido por regras simples.

## 1. Importacao de bibliotecas

As bibliotecas abaixo sao suficientes para carregar os dados, normalizar o texto e consolidar os resultados em tabela.

In [ ]:
from collections import Counter
from pathlib import Path
import re
import unicodedata

import pandas as pd

## 2. Leitura dos arquivos da atividade

Nesta celula, carregamos:
- o arquivo `.txt` com os relatos dos pacientes;
- o arquivo `.csv` com o mapa de conhecimento entre sintomas e diagnosticos.

In [ ]:
# Resolve o diretorio base tanto no VS Code quanto ao abrir o notebook pela pasta notebooks.
BASE_DIR = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
FRASES_PATH = BASE_DIR / 'data' / 'frases_sintomas_pacientes.txt'
MAPA_PATH = BASE_DIR / 'data' / 'mapa_conhecimento_sintomas_doencas.csv'

# Carrega apenas linhas nao vazias do arquivo de frases.
with open(FRASES_PATH, encoding='utf-8') as arquivo:
    frases = [linha.strip() for linha in arquivo if linha.strip()]

# Carrega o mapa de conhecimento com pandas para facilitar a consulta.
mapa = pd.read_csv(MAPA_PATH)

print(f'Total de frases: {len(frases)}')
mapa.head()

## 3. Funcoes principais

A logica do diagnostico assistido foi separada em duas funcoes:
- `normalizar_texto`: remove acentos, pontuacao e padroniza o texto;
- `identificar_diagnosticos`: compara a frase com os sintomas cadastrados no mapa.

In [ ]:
def normalizar_texto(texto: str) -> str:
    # Remove acentos para facilitar a comparacao entre textos com grafias diferentes.
    texto_sem_acentos = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('ascii')
    # Remove pontuacao e padroniza tudo em letras minusculas.
    texto_limpo = re.sub(r'[^a-zA-Z0-9\s]', ' ', texto_sem_acentos.lower())
    # Elimina espacos repetidos gerados pela limpeza.
    return re.sub(r'\s+', ' ', texto_limpo).strip()


def identificar_diagnosticos(frase: str, mapa_conhecimento: pd.DataFrame) -> dict:
    frase_normalizada = normalizar_texto(frase)
    correspondencias = []

    # Percorre cada linha do mapa para buscar sintomas presentes na frase.
    for linha in mapa_conhecimento.itertuples(index=False):
        sintomas = []
        for coluna in ('sintoma_1', 'sintoma_2'):
            valor = getattr(linha, coluna)
            if isinstance(valor, str) and valor.strip():
                sintomas.append(normalizar_texto(valor))

        sintomas_encontrados = [s for s in sintomas if s and s in frase_normalizada]
        if sintomas_encontrados:
            correspondencias.append((linha.doenca_associada, sintomas_encontrados))

    # Caso nenhum sintoma seja localizado, a frase e encaminhada para avaliacao clinica.
    if not correspondencias:
        return {
            'frase': frase,
            'sintomas_identificados': 'nenhum sintoma mapeado',
            'diagnostico_sugerido': 'Encaminhar para avaliacao clinica',
            'confianca_regra': 0,
        }

    # Conta quantas vezes cada doenca apareceu nas correspondencias.
    contagem = Counter(item[0] for item in correspondencias)
    diagnostico_sugerido, confianca = contagem.most_common(1)[0]
    sintomas_unicos = sorted({sintoma for _, lista in correspondencias for sintoma in lista})

    return {
        'frase': frase,
        'sintomas_identificados': ', '.join(sintomas_unicos),
        'diagnostico_sugerido': diagnostico_sugerido,
        'confianca_regra': confianca,
    }

## 4. Aplicacao da regra nas frases

A seguir, o notebook processa todas as frases da base simulada e monta uma tabela final com sintomas identificados e diagnostico sugerido.

In [ ]:
# Aplica a funcao em cada frase e converte o resultado final para DataFrame.
resultados = [identificar_diagnosticos(frase, mapa) for frase in frases]
df_resultados = pd.DataFrame(resultados)
df_resultados

## 5. Resumo dos diagnosticos sugeridos

Este resumo ajuda a visualizar quais diagnosticos apareceram com mais frequencia no conjunto de frases.

In [ ]:
# Conta quantas vezes cada diagnostico foi sugerido nas frases analisadas.
df_resultados['diagnostico_sugerido'].value_counts()

## 6. Interpretacao final

A logica aplicada aqui e baseada em regras simples de correspondencia textual.

Pontos fortes:
- facil de explicar na apresentacao;
- transparente para fins academicos;
- adequada para demonstrar uma ontologia inicial.

Limitacoes:
- nao entende contexto clinico complexo;
- depende muito das palavras cadastradas no CSV;
- nao substitui avaliacao medica.